# PM2.5 city time-series vs. the WHO guideline

Fetch a **year** of daily-rollup PM2.5 for one city, plot the series, overlay the **WHO 24-hour guideline (15 µg/m³)**, and annotate the exceedance days. Shows the value of OpenAQ's server-side daily rollup for long windows — a year is ~365 rows per sensor, not tens of thousands of raw readings.

> Needs a free `OPENAQ_API_KEY` (see the authentication page). The live cell is guarded so the notebook stays `--nbval-lax`-safe.

In [ ]:
import os
from earthlens import EarthLens

In [ ]:
WHO_24H_GUIDELINE = 15.0  # µg/m³, WHO 2021 24-hour PM2.5 guideline
bbox_lat = [34.0, 34.3]   # Los Angeles basin
bbox_lon = [-118.5, -118.1]
start, end = "2023-01-01", "2023-12-31"

In [ ]:
df = None
if os.environ.get("OPENAQ_API_KEY"):
    df = EarthLens(
        data_source="openaq",
        variables=["pm25"],
        start=start, end=end,
        aoi=[bbox_lon[0], bbox_lat[0], bbox_lon[1], bbox_lat[1]],
        temporal_resolution="daily",   # server-side daily rollup
        max_locations=5,
        path="out/openaq",
    ).download(progress_bar=False)
    print(df.shape)
else:
    print("set OPENAQ_API_KEY to run the live cell")

In [ ]:
# Collapse to a city-wide daily mean across the stations.
city = None
if df is not None and not df.empty:
    city = (
        df.assign(day=df["datetime_utc"].dt.floor("D"))
          .groupby("day")["value"].mean()
          .sort_index()
    )
    exceedances = city[city > WHO_24H_GUIDELINE]
    print(f"{len(exceedances)} of {len(city)} days exceed the WHO guideline")

In [ ]:
if city is not None and not city.empty:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(city.index, city.values, color="#1f77b4", lw=1, label="daily mean PM2.5")
    ax.axhline(WHO_24H_GUIDELINE, color="crimson", ls="--", label="WHO 24h guideline (15)")
    ax.scatter(exceedances.index, exceedances.values, color="crimson", s=12, zorder=3, label="exceedance day")
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.set_title("Los Angeles daily PM2.5 vs. WHO guideline, 2023")
    ax.legend(fontsize=8)
    fig.autofmt_xdate()
    plt.show()